# Spectral function and Green's-function determinant

Two ways of looking at the same correlated Green's function, on the two classic four- and
six-electron $\pi$ systems:

- the **spectral function** $-\mathrm{Im}\,\mathrm{Tr}\,G(\omega)$, the usual density of
  states, which peaks at the **poles** of $G$;
- $\log|\det G(\omega)|$, which peaks at those same poles but *also* dips at the **zeros**
  of $\det G$.

The zeros carry no spectral weight, so the first picture is completely blind to them — yet
they are half of what determines the topology of $G$ (see
[`docs/theory.md`](../docs/theory.md)). Switching the two-electron interaction off, with
everything else held fixed, makes the point: without it, $\det G$ has poles and no zeros at
all.

Everything here is generated from coordinates written in this notebook, and runs in a few
seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyscf import gto

from casgf import ActiveSpace, lehmann

FREQS = np.linspace(-1.5, 1.5, 1201)
ETA = 0.01

In [ ]:
CH = 1.08          # C-H bond length, Angstrom
PERIMETER = 2.90   # a + b held fixed along the scan


def cyclobutadiene(a, b=None):
    """Planar cyclobutadiene with C-C bonds of alternating length `a` and `b`.

    Carbons sit on the corners of a rectangle; each C-H bond runs along the
    exterior angle bisector, which is the ring diagonal. `a == b` is the square
    D4h transition state of the automerization; the D2h ground state has
    alternating short and long bonds.

    Idealised, not optimised -- good enough to show the physics, and it keeps
    the notebook self-contained.
    """
    b = PERIMETER - a if b is None else b
    atoms = []
    for sx, sy in ((1, 1), (-1, 1), (-1, -1), (1, -1)):
        cx, cy = sx * a / 2, sy * b / 2
        atoms.append(f"C {cx:.8f} {cy:.8f} 0.0")
        atoms.append(f"H {cx + sx * CH / np.sqrt(2):.8f} {cy + sy * CH / np.sqrt(2):.8f} 0.0")
    return "; ".join(atoms)


def benzene(cc=1.39, ch=1.09):
    """Idealised D6h benzene: a regular hexagon of carbons with radial C-H bonds."""
    atoms = []
    for k in range(6):
        c, s = np.cos(2 * np.pi * k / 6), np.sin(2 * np.pi * k / 6)
        atoms.append(f"C {cc * c:.8f} {cc * s:.8f} 0.0")
        atoms.append(f"H {(cc + ch) * c:.8f} {(cc + ch) * s:.8f} 0.0")
    return "; ".join(atoms)

In [ ]:
def with_and_without_interaction(atom, ncas, nelecas, basis="def2-SVP"):
    """CASSCF, then the Green's function of the full and of the one-body Hamiltonian.

    The non-interacting case reuses the *same* CASSCF orbitals and the same h1; only
    the two-electron term is zeroed. Both are referenced to their own particle-hole
    symmetric chemical potential, so both sit centred on the middle of their gap and
    the two curves can be read on one axis.
    """
    mol = gto.M(atom=atom, basis=basis, unit="A", verbose=0)
    space = ActiveSpace.from_molecule(mol, ncas=ncas, nelecas=nelecas)
    free = ActiveSpace.from_arrays(space.h1, np.zeros_like(space.eri), space.nelecas)
    print(f"CASSCF({nelecas},{ncas})/{basis} = {space.meta['e_tot']:.8f} Ha")
    return {"interacting": lehmann(space), "non-interacting": lehmann(free)}


def plot(curves, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
    for label, gf in curves.items():
        axes[0].plot(FREQS, gf.spectral(FREQS, ETA), label=label)
        axes[1].plot(FREQS, gf.log_abs_det(FREQS, ETA), label=label)
    axes[0].set_ylabel(r"$-\mathrm{Im}\,\mathrm{Tr}\,G(\omega)$")
    axes[1].set_ylabel(r"$\log|\det G(\omega)|$")
    for ax in axes:
        ax.set_xlabel(r"$\omega$ (a.u.)")
        ax.axvline(0, color="k", lw=0.5, ls="--")
        ax.legend(fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def summarise(curves):
    for label, gf in curves.items():
        print(f"  {label:16s} mu = {gf.mu:+.6f}   gap = {gf.gap:.6f}"
              f"   sum rule = {gf.sum_rule():.9f}")

## Cyclobutadiene — CAS(4,4)/def2-SVP

The four $\pi$ electrons in the four $\pi$ orbitals of the rectangular D2h ground state.
Antiaromatic and strongly correlated: a single determinant is a poor description, which is
why it is the standard small test case for multireference methods.

In [ ]:
cbd = with_and_without_interaction(cyclobutadiene(1.34), ncas=4, nelecas=4)
summarise(cbd)
plot(cbd, "Cyclobutadiene (rectangular), CAS(4,4)/def2-SVP")

## Benzene — CAS(6,6)/def2-SVP

The aromatic counterpart: six $\pi$ electrons in six orbitals, and a much wider gap.

In [ ]:
bz = with_and_without_interaction(benzene(), ncas=6, nelecas=6)
summarise(bz)
plot(bz, "Benzene, CAS(6,6)/def2-SVP")

## What to look at

In the right-hand panels the interacting curve dips where the non-interacting one does not.
Those dips are zeros of $\det G$, and they exist only because of the two-electron
interaction. The left-hand panels — the ordinary spectral function — show no trace of them.

The sum rule printed above is a check rather than a result: the total spectral weight has
to equal the number of active orbitals exactly, and it does, to nine digits.